In [12]:
from typing import Literal
import pandas as pd
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
import inspect
import preprocessing.column_calculator as cc

In [13]:
df=pd.read_csv("data/last3months.csv",delimiter=";")
df

,switch_code,banque_emetrice,fraud_response_code,id_magasin,imei,is_international,source,de43_card_acceptor_name,de42_aceptor_merchant_code,de41_acceptor_terminal_id,...,de45_track1data,de35_track2data,de36_track3data,de30_trans_amount_proc_fees,de28_transaction_amount_fees,de49_trans_curr_code,de104_transaction_description,de07_transmission_date_time,transaction_autorization,transaction_status
0,++jVXa2070mcXVs/I6CXTw==,BNA,NaN,10044.0,V9E0039925,0,POS,NaN,10115,182169011,...,NaN,5321360000922483=27056211881501300000,NaN,NaN,NaN,788.0,NaN,NaN,Online,Transaction autorisée
1,++mgmIcGikSgtfm8Rsniig==,ABT,NaN,10036.0,V9E0039929,0,POS,NaN,10103,182099010,...,NaN,4938364025013649=27096011798700000000,NaN,NaN,NaN,788.0,NaN,NaN,Online,Transaction autorisée
2,++MtXnbnxU2kpDNKP9HN+Q==,STB,NaN,10037.0,V9E0040123,0,POS,NaN,10104,182109017,...,NaN,5399970001283110=24076211741423400000,NaN,NaN,NaN,788.0,NaN,NaN,Online,Transaction autorisée
3,++z5soCawk2KDb2aX5VIug==,AMEN BANK,NaN,10034.0,V9E0039920,0,POS,NaN,10099,182079012,...,NaN,4146297700001364=26066011593411500000,NaN,NaN,NaN,788.0,NaN,NaN,Online,Transaction autorisée
4,+/6p9a26NkGD/Z86g1jCoA==,ONP,NaN,10046.0,V9E0039932,0,POS,NaN,10114,182209015,...,NaN,5359401419421835=24066213524259400000,NaN,NaN,NaN,788.0,NaN,NaN,Online,Transaction autorisée
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1402,zu6c5HQi/k27Mb5X7tEtMg==,ABT,NaN,10051.0,V9E0040126,0,POS,NaN,10123,182249011,...,NaN,5494904017488839=27042011950774200000,NaN,NaN,NaN,788.0,NaN,NaN,Online,Transaction autorisée
1403,ZWqanxOHBEytJ3fmTuG/Vw==,BIAT,NaN,10031.0,V9E0030556,0,POS,NaN,10092,182049015,...,NaN,5420262827737385=23106011044883300000,NaN,NaN,NaN,788.0,NaN,NaN,Online,Transaction non autorisée
1404,zYZH4M5C1UGE/Iy/ti128g==,STUSID,NaN,10031.0,V9E0030556,0,POS,NaN,10092,182049015,...,NaN,5129790099003064=25096011737021900000,NaN,NaN,NaN,788.0,NaN,NaN,Online,Transaction non autorisée
1405,ZZ3qtTyEsk++/sJLgbHlWA==,BT,NaN,10036.0,V9E0039929,0,POS,NaN,10103,182099010,...,NaN,4557690100275921=27016211538479300000,NaN,NaN,NaN,788.0,NaN,NaN,Online,Transaction autorisée


In [14]:
df["requestdate"]=pd.to_datetime(df["requestdate"])

In [15]:
def authorization_refund_count_and_value(
    df_max_window: pd.DataFrame,
    window_sizes: tuple[str],
    entity: Literal["customer", "merchant", "terminal"],
) -> pd.DataFrame:
    filtered_df = df_max_window
    filtered_df["is_refund"]=(filtered_df["de39_response_code"] == 0) & (filtered_df["de03_processing_code"] == 20)
    filtered_df["value_refund"]=filtered_df["de04_amount_transaction"]*filtered_df["is_refund"]
    for window_size in window_sizes:
        
        number_of_refunds = (
            filtered_df["is_refund"]
            .rolling(window_size)
            .sum()
        )
        value_of_refunds = (
            filtered_df["value_refund"]
            .rolling(window_size)
            .sum()
        )
        filtered_df[f"{entity}_authorization_refund_count_{window_size}"] = (
            number_of_refunds.to_list()
        )
        filtered_df[f"{entity}_authorization_refund_value_{window_size}"] = (
            value_of_refunds.to_list()
        )
    filtered_df=filtered_df.drop(columns=["is_refund","value_refund"])
    return filtered_df

def test(
    df_max_window: pd.DataFrame,
    window_sizes: tuple[str],
    entity: Literal["card", "merchant", "terminal"],):
    df_max_window.set_index("requestdate", inplace=True)
    df_max_window.sort_index(inplace=True)
    df_max_window=authorization_refund_count_and_value(df_max_window,window_sizes,entity)
    df_max_window.reset_index(inplace=True)
    return df_max_window

transformed_data = df.groupby("de42_aceptor_merchant_code").apply(
            lambda x: test(x,window_sizes=["15min"],entity="merchant")
        )
        # Reset the index and drop the 'commercant' column
transformed_data = transformed_data.reset_index(drop=True)
transformed_data

/tmp/ipykernel_31151/2602594533.py:40: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  transformed_data = df.groupby("de42_aceptor_merchant_code").apply(


,requestdate,switch_code,banque_emetrice,fraud_response_code,id_magasin,imei,is_international,source,de43_card_acceptor_name,de42_aceptor_merchant_code,...,de36_track3data,de30_trans_amount_proc_fees,de28_transaction_amount_fees,de49_trans_curr_code,de104_transaction_description,de07_transmission_date_time,transaction_autorization,transaction_status,merchant_authorization_refund_count_15min,merchant_authorization_refund_value_15min
0,2024-04-02,h8nRum80uEWjcUgsuXD5Iw==,ABT,NaN,10031.0,V9E0030556,0,POS,NaN,10092,...,NaN,NaN,NaN,788.0,NaN,NaN,Online,Transaction annulée,0.0,0.0
1,2024-04-02,6eJfE165yk22tkqR13/b1Q==,ABT,NaN,10031.0,V9E0030556,0,POS,NaN,10092,...,NaN,NaN,NaN,788.0,NaN,NaN,Online,Transaction autorisée,0.0,0.0
2,2024-04-02,ZS7Vv7LPb0mGq82ytM0nIg==,ABT,NaN,10031.0,V9E0030556,0,POS,NaN,10092,...,NaN,NaN,NaN,788.0,NaN,NaN,Online,Transaction autorisée,0.0,0.0
3,2024-04-02,VkdjKohDAUCPSwHBcohYWA==,QNB,NaN,10031.0,V9E0030556,0,POS,NaN,10092,...,NaN,NaN,NaN,788.0,NaN,NaN,Online,Transaction autorisée,0.0,0.0
4,2024-04-02,Ty/vE2gbZkChUS5onLRkgQ==,ABT,NaN,10031.0,V9E0030556,0,POS,NaN,10092,...,NaN,NaN,NaN,788.0,NaN,NaN,Online,Transaction autorisée,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1402,2024-06-25,UuxtHkl8NkmQm4pY/LxZXQ==,ABT,NaN,10051.0,V9E0040126,0,POS,NaN,10123,...,NaN,NaN,NaN,788.0,NaN,NaN,Online,Transaction autorisée,0.0,0.0
1403,2024-06-25,ihhp7v589EuEya1VQ+PTSQ==,BIAT,NaN,10051.0,V9E0040126,0,POS,NaN,10123,...,NaN,NaN,NaN,788.0,NaN,NaN,Online,Transaction autorisée,0.0,0.0
1404,2024-06-25,M9rvFXUpdUajuMFQRPUEgA==,ATB,NaN,10051.0,V9E0040126,0,POS,NaN,10123,...,NaN,NaN,NaN,788.0,NaN,NaN,Online,Transaction autorisée,0.0,0.0
1405,2024-06-25,9lfEqwHJjEmulATF7sfKsQ==,BIAT,NaN,10051.0,V9E0040126,0,POS,NaN,10123,...,NaN,NaN,NaN,788.0,NaN,NaN,Online,Transaction autorisée,0.0,0.0


In [16]:
transformed_data

,requestdate,switch_code,banque_emetrice,fraud_response_code,id_magasin,imei,is_international,source,de43_card_acceptor_name,de42_aceptor_merchant_code,...,de36_track3data,de30_trans_amount_proc_fees,de28_transaction_amount_fees,de49_trans_curr_code,de104_transaction_description,de07_transmission_date_time,transaction_autorization,transaction_status,merchant_authorization_refund_count_15min,merchant_authorization_refund_value_15min
0,2024-04-02,h8nRum80uEWjcUgsuXD5Iw==,ABT,NaN,10031.0,V9E0030556,0,POS,NaN,10092,...,NaN,NaN,NaN,788.0,NaN,NaN,Online,Transaction annulée,0.0,0.0
1,2024-04-02,6eJfE165yk22tkqR13/b1Q==,ABT,NaN,10031.0,V9E0030556,0,POS,NaN,10092,...,NaN,NaN,NaN,788.0,NaN,NaN,Online,Transaction autorisée,0.0,0.0
2,2024-04-02,ZS7Vv7LPb0mGq82ytM0nIg==,ABT,NaN,10031.0,V9E0030556,0,POS,NaN,10092,...,NaN,NaN,NaN,788.0,NaN,NaN,Online,Transaction autorisée,0.0,0.0
3,2024-04-02,VkdjKohDAUCPSwHBcohYWA==,QNB,NaN,10031.0,V9E0030556,0,POS,NaN,10092,...,NaN,NaN,NaN,788.0,NaN,NaN,Online,Transaction autorisée,0.0,0.0
4,2024-04-02,Ty/vE2gbZkChUS5onLRkgQ==,ABT,NaN,10031.0,V9E0030556,0,POS,NaN,10092,...,NaN,NaN,NaN,788.0,NaN,NaN,Online,Transaction autorisée,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1402,2024-06-25,UuxtHkl8NkmQm4pY/LxZXQ==,ABT,NaN,10051.0,V9E0040126,0,POS,NaN,10123,...,NaN,NaN,NaN,788.0,NaN,NaN,Online,Transaction autorisée,0.0,0.0
1403,2024-06-25,ihhp7v589EuEya1VQ+PTSQ==,BIAT,NaN,10051.0,V9E0040126,0,POS,NaN,10123,...,NaN,NaN,NaN,788.0,NaN,NaN,Online,Transaction autorisée,0.0,0.0
1404,2024-06-25,M9rvFXUpdUajuMFQRPUEgA==,ATB,NaN,10051.0,V9E0040126,0,POS,NaN,10123,...,NaN,NaN,NaN,788.0,NaN,NaN,Online,Transaction autorisée,0.0,0.0
1405,2024-06-25,9lfEqwHJjEmulATF7sfKsQ==,BIAT,NaN,10051.0,V9E0040126,0,POS,NaN,10123,...,NaN,NaN,NaN,788.0,NaN,NaN,Online,Transaction autorisée,0.0,0.0


In [17]:
test_df=pd.DataFrame({"A":[1,2,3],
                      "B":[1,1,1]})

In [18]:
codes=df["de39_response_code"].unique()
codes=[int(code) for code in codes[(~np.isnan(codes)) & (codes>0)]]
len(codes)

12

In [19]:
DEFAULT_WINDOWS = ["60min", "24h"]
ENTITY_IDS = {
    "terminal": "de41_acceptor_terminal_id",
    "merchant": "de42_aceptor_merchant_code",
    "card": "de02_pan",
}


In [20]:
class column_calculator(BaseEstimator, TransformerMixin):
    def __init__(
        self,
        metrics: dict,
        entity: Literal["card", "merchant", "terminal", "bin"],
    ):
        """
        Initializes the entitySpendingBehaviorFeatures transformer.

        Args:
            metrics (dict, optional): a dictionary containing necessary information for each mastercard rule. Defaults to rule_set.
        """
        self.entity = entity
        self.metrics = metrics

    def fit(self, X, y=None):
        # nothing to do here
        return self

    def calculate_columns(self,df_max_window,window_sizes,entity,metrics,metrics_funcs):
        transformed_data=df_max_window
        transformed_data.set_index("requestdate", inplace=True)
        transformed_data.sort_index(inplace=True)
        for metric in self.metrics:
            func = metrics_funcs.get(metric)
            if func:
                transformed_data = func(
                    df_max_window=df_max_window,
                    window_sizes=self.metrics[metric]["windows"],
                    entity=self.entity,
                )
        transformed_data.reset_index(inplace=True)

        return transformed_data

    def transform(self, X) -> pd.DataFrame:
        transformed_data = X

        metrics_funcs = {
                name: func for name, func in inspect.getmembers(cc, inspect.isfunction)
            }
        

        transformed_data = X  # Initialize with X or the initial data
        transformed_data=transformed_data.groupby(ENTITY_IDS.get(self.entity)).apply(lambda x:self.calculate_columns(df_max_window=x,window_sizes=DEFAULT_WINDOWS,entity=self.entity,metrics=self.metrics,metrics_funcs=metrics_funcs))
        transformed_data = transformed_data.reset_index(drop=True)

        return transformed_data

In [21]:
def transform_dataframe(
    df_max_window: pd.DataFrame,
    entity: Literal["card", "merchant", "terminal"],
):
    """Calculate entity metrics

    Args:
        df_max_window (pd.DataFrame):_description_
        processing_time (pd._libs.tslibs.timestamps.Timestamp):_description_
        metrics (dict, optional):_description_. Defaults to { "number_of_auth_transactions": {"windows": DEFAULT_WINDOWS}, }.
        Those are the requestes metrics for the entitys with values the time windows.

    Returns:
        pd.DataFrame: The new dataframe with the calculated metrics.
    """
    all_metrics = [
        metric[0]
        for metric in inspect.getmembers(cc, inspect.isfunction)
        if metric[0] not in ["merchantSpatialTimeFeatures"]
    ]

    metrics_with_default_windows = dict(
        zip(all_metrics, len(all_metrics) * [{"windows": DEFAULT_WINDOWS}])
    )
    entitys_metrics_calculator = column_calculator(
        metrics=metrics_with_default_windows,
        entity=entity,
    )

    return entitys_metrics_calculator.transform(df_max_window)

In [22]:
tdf=transform_dataframe(df,"merchant")
tdf

/tmp/ipykernel_31151/2845296641.py:45: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  transformed_data=transformed_data.groupby(ENTITY_IDS.get(self.entity)).apply(lambda x:self.calculate_columns(df_max_window=x,window_sizes=DEFAULT_WINDOWS,entity=self.entity,metrics=self.metrics,metrics_funcs=metrics_funcs))


,requestdate,switch_code,banque_emetrice,fraud_response_code,id_magasin,imei,is_international,source,de43_card_acceptor_name,de42_aceptor_merchant_code,...,merchant_technical_falback_count_60min,merchant_ratiotechnical_falback_60min,merchant_technical_falback_count_24h,merchant_ratiotechnical_falback_24h,merchant_count_response_code_810_60min,merchant_count_response_code_810_24h,merchant_count_response_code_107_60min,merchant_count_response_code_107_24h,merchant_count_response_code_55_60min,merchant_count_response_code_55_24h
0,2024-04-02,h8nRum80uEWjcUgsuXD5Iw==,ABT,NaN,10031.0,V9E0030556,0,POS,NaN,10092,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
1,2024-04-02,6eJfE165yk22tkqR13/b1Q==,ABT,NaN,10031.0,V9E0030556,0,POS,NaN,10092,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
2,2024-04-02,ZS7Vv7LPb0mGq82ytM0nIg==,ABT,NaN,10031.0,V9E0030556,0,POS,NaN,10092,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
3,2024-04-02,VkdjKohDAUCPSwHBcohYWA==,QNB,NaN,10031.0,V9E0030556,0,POS,NaN,10092,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
4,2024-04-02,Ty/vE2gbZkChUS5onLRkgQ==,ABT,NaN,10031.0,V9E0030556,0,POS,NaN,10092,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1402,2024-06-25,UuxtHkl8NkmQm4pY/LxZXQ==,ABT,NaN,10051.0,V9E0040126,0,POS,NaN,10123,...,0.0,0.0,0.0,0.0,1.0,1.0,NaN,NaN,0.0,0.0
1403,2024-06-25,ihhp7v589EuEya1VQ+PTSQ==,BIAT,NaN,10051.0,V9E0040126,0,POS,NaN,10123,...,0.0,0.0,0.0,0.0,1.0,1.0,NaN,NaN,0.0,0.0
1404,2024-06-25,M9rvFXUpdUajuMFQRPUEgA==,ATB,NaN,10051.0,V9E0040126,0,POS,NaN,10123,...,0.0,0.0,0.0,0.0,1.0,1.0,NaN,NaN,0.0,0.0
1405,2024-06-25,9lfEqwHJjEmulATF7sfKsQ==,BIAT,NaN,10051.0,V9E0040126,0,POS,NaN,10123,...,0.0,0.0,0.0,0.0,1.0,1.0,NaN,NaN,0.0,0.0


In [23]:
new_df=pd.read_csv("data/switch.csv",delimiter=";")
new_df["requestdate"]=pd.to_datetime(new_df["requestdate"])
new_df

,switch_code,banque_emetrice,fraud_response_code,id_magasin,imei,is_international,source,de43_card_acceptor_name,de42_aceptor_merchant_code,de41_acceptor_terminal_id,...,de45_track1data,de35_track2data,de36_track3data,de30_trans_amount_proc_fees,de28_transaction_amount_fees,de49_trans_curr_code,de104_transaction_description,de07_transmission_date_time,transaction_autorization,transaction_status
0,++gR+DKuhkquZNNUGkVg9g==,BIAT,NaN,10034.0,V9E0039920,0,POS,NaN,10099,182079012,...,NaN,5218730914969828=25026011796430600000,NaN,NaN,NaN,788.0,NaN,NaN,Online,Transaction autorisée
1,++jVXa2070mcXVs/I6CXTw==,BNA,NaN,10044.0,V9E0039925,0,POS,NaN,10115,182169011,...,NaN,5321360000922483=27056211881501300000,NaN,NaN,NaN,788.0,NaN,NaN,Online,Transaction autorisée
2,++mgmIcGikSgtfm8Rsniig==,ABT,NaN,10036.0,V9E0039929,0,POS,NaN,10103,182099010,...,NaN,4938364025013649=27096011798700000000,NaN,NaN,NaN,788.0,NaN,NaN,Online,Transaction autorisée
3,++MtXnbnxU2kpDNKP9HN+Q==,STB,NaN,10037.0,V9E0040123,0,POS,NaN,10104,182109017,...,NaN,5399970001283110=24076211741423400000,NaN,NaN,NaN,788.0,NaN,NaN,Online,Transaction autorisée
4,++z5soCawk2KDb2aX5VIug==,AMEN BANK,NaN,10034.0,V9E0039920,0,POS,NaN,10099,182079012,...,NaN,4146297700001364=26066011593411500000,NaN,NaN,NaN,788.0,NaN,NaN,Online,Transaction autorisée
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2635,Zwu+p48nxUmJPZqrL7BMlg==,BT,NaN,10039.0,V9E0039923,0,POS,NaN,10109,182129015,...,NaN,4557690100839023=25096211453992600000,NaN,NaN,NaN,788.0,NaN,NaN,Online,Transaction autorisée
2636,ZxoLX6CaE0elXQAuJWEkVg==,BT,NaN,10036.0,V9E0039929,0,POS,NaN,10103,182099010,...,NaN,5399909900730508=24106211766227100000,NaN,NaN,NaN,788.0,NaN,NaN,Online,Transaction autorisée
2637,zYZH4M5C1UGE/Iy/ti128g==,STUSID,NaN,10031.0,V9E0030556,0,POS,NaN,10092,182049015,...,NaN,5129790099003064=25096011737021900000,NaN,NaN,NaN,788.0,NaN,NaN,Online,Transaction non autorisée
2638,ZZ3qtTyEsk++/sJLgbHlWA==,BT,NaN,10036.0,V9E0039929,0,POS,NaN,10103,182099010,...,NaN,4557690100275921=27016211538479300000,NaN,NaN,NaN,788.0,NaN,NaN,Online,Transaction autorisée


In [24]:
from preprocessing.preprocessing import transform_dataframe

transformed_df=transform_dataframe(new_df,"merchant")
transformed_df

,requestdate,switch_code,banque_emetrice,fraud_response_code,id_magasin,imei,is_international,source,de43_card_acceptor_name,de42_aceptor_merchant_code,...,merchant_count_response_code_810_24h,merchant_count_response_code_107_24h,merchant_count_response_code_800_60min,merchant_count_response_code_800_24h,merchant_count_response_code_201_60min,merchant_count_response_code_56_60min,merchant_count_response_code_201_24h,merchant_count_response_code_56_24h,merchant_count_response_code_55_60min,merchant_count_response_code_55_24h
0,2024-04-02 16:01:19,Ktmx0bQGkUKpEjv2Wqe3Pw==,BIAT,NaN,10031.0,V9E0030556,0,POS,NaN,10092,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2024-04-02 16:02:59,Qm9XVf8vDkizP6R3Jphf9g==,ABT,NaN,10031.0,V9E0030556,0,POS,NaN,10092,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2024-04-02 16:04:13,ZS7Vv7LPb0mGq82ytM0nIg==,ABT,NaN,10031.0,V9E0030556,0,POS,NaN,10092,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2024-04-02 16:05:13,VkdjKohDAUCPSwHBcohYWA==,QNB,NaN,10031.0,V9E0030556,0,POS,NaN,10092,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2024-04-02 16:07:05,Ty/vE2gbZkChUS5onLRkgQ==,ABT,NaN,10031.0,V9E0030556,0,POS,NaN,10092,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2635,2024-07-10 23:04:38,qq+9nNtkI0CXhRp3s+HgtQ==,ATB,NaN,10058.0,V9E0040088,0,POS,NaN,10134,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2636,2024-07-10 23:33:21,Af0F/gPvbEOzErTW4GMcHA==,AMEN BANK,NaN,10058.0,V9E0040088,0,POS,NaN,10134,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2637,2024-07-11 01:58:56,lCuB7YNGakOnQ89E6Z5JQA==,NaN,NaN,10058.0,V9E0040088,1,POS,NaN,10134,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2638,2024-07-11 01:59:25,xv23UgHuQkKJyvpdLyPLFA==,NaN,NaN,10058.0,V9E0040088,1,POS,NaN,10134,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [25]:
df1=transformed_df[transformed_df["de42_aceptor_merchant_code"]==10092]
df1[df1["de39_response_code"].isna()]

,requestdate,switch_code,banque_emetrice,fraud_response_code,id_magasin,imei,is_international,source,de43_card_acceptor_name,de42_aceptor_merchant_code,...,merchant_count_response_code_810_24h,merchant_count_response_code_107_24h,merchant_count_response_code_800_60min,merchant_count_response_code_800_24h,merchant_count_response_code_201_60min,merchant_count_response_code_56_60min,merchant_count_response_code_201_24h,merchant_count_response_code_56_24h,merchant_count_response_code_55_60min,merchant_count_response_code_55_24h
41,2024-04-29 10:06:27,c/plI5R0a061sT2UR0CatA==,STUSID,NaN,10031.0,V9E0030556,0,POS,NaN,10092,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
42,2024-04-29 10:06:59,u6hJMxciiEu4O6arzsCN6w==,STUSID,NaN,10031.0,V9E0030556,0,POS,NaN,10092,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
80,2024-05-08 15:42:03,QuXnN989FUuedMzOUurPKQ==,NaN,NaN,10031.0,V9E0019593,1,POS,NaN,10092,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
84,2024-05-10 11:11:11,1PQ3dhsJkUyZjaj1emm8aA==,BIAT,NaN,10031.0,V9E0030555,0,POS,NaN,10092,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
85,2024-05-10 11:14:53,vyWuBUz0kEWJLl8GrRmOBA==,ONP,NaN,10031.0,V9E0030555,0,POS,NaN,10092,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
87,2024-05-10 11:21:34,tRn5lSY8V0m+r8bmHeb3Eg==,BIAT,NaN,10031.0,V9E0030555,0,POS,NaN,10092,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
88,2024-05-10 11:22:33,cVjC6Pr2KkyPxP3ZoL6Wzg==,BIAT,NaN,10031.0,V9E0030555,0,POS,NaN,10092,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
89,2024-05-10 11:23:14,qg9nPbo/0020JaABDNuRWw==,ONP,NaN,10031.0,V9E0030555,0,POS,NaN,10092,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
90,2024-05-10 11:26:01,ISfFpx+GyEGGE1f3xVfqIw==,ABT,NaN,10031.0,V9E0030555,0,POS,NaN,10092,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
91,2024-05-10 11:27:18,Gm0uHsJU9EmTE67eaTgImQ==,ABT,NaN,10031.0,V9E0030555,0,POS,NaN,10092,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [26]:
pd.set_option('display.max_columns', None)

In [27]:
american_df=pd.read_csv("data/fraudTrain.csv")
american_df

,Unnamed: 0,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,city,state,zip,lat,long,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud
0,0,2019-01-01 00:00:18,2703186189652095,"fraud_Rippin, Kub and Mann",misc_net,4.97,Jennifer,Banks,F,561 Perry Cove,Moravian Falls,NC,28654,36.0788,-81.1781,3495,"Psychologist, counselling",1988-03-09,0b242abb623afc578575680df30655b9,1325376018,36.011293,-82.048315,0
1,1,2019-01-01 00:00:44,630423337322,"fraud_Heller, Gutmann and Zieme",grocery_pos,107.23,Stephanie,Gill,F,43039 Riley Greens Suite 393,Orient,WA,99160,48.8878,-118.2105,149,Special educational needs teacher,1978-06-21,1f76529f8574734946361c461b024d99,1325376044,49.159047,-118.186462,0
2,2,2019-01-01 00:00:51,38859492057661,fraud_Lind-Buckridge,entertainment,220.11,Edward,Sanchez,M,594 White Dale Suite 530,Malad City,ID,83252,42.1808,-112.2620,4154,Nature conservation officer,1962-01-19,a1a22d70485983eac12b5b88dad1cf95,1325376051,43.150704,-112.154481,0
3,3,2019-01-01 00:01:16,3534093764340240,"fraud_Kutch, Hermiston and Farrell",gas_transport,45.00,Jeremy,White,M,9443 Cynthia Court Apt. 038,Boulder,MT,59632,46.2306,-112.1138,1939,Patent attorney,1967-01-12,6b849c168bdad6f867558c3793159a81,1325376076,47.034331,-112.561071,0
4,4,2019-01-01 00:03:06,375534208663984,fraud_Keeling-Crist,misc_pos,41.96,Tyler,Garcia,M,408 Bradley Rest,Doe Hill,VA,24433,38.4207,-79.4629,99,Dance movement psychotherapist,1986-03-28,a41d7549acf90789359a9aa5346dcb46,1325376186,38.674999,-78.632459,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1296670,1296670,2020-06-21 12:12:08,30263540414123,fraud_Reichel Inc,entertainment,15.56,Erik,Patterson,M,162 Jessica Row Apt. 072,Hatch,UT,84735,37.7175,-112.4777,258,Geoscientist,1961-11-24,440b587732da4dc1a6395aba5fb41669,1371816728,36.841266,-111.690765,0
1296671,1296671,2020-06-21 12:12:19,6011149206456997,fraud_Abernathy and Sons,food_dining,51.70,Jeffrey,White,M,8617 Holmes Terrace Suite 651,Tuscarora,MD,21790,39.2667,-77.5101,100,"Production assistant, television",1979-12-11,278000d2e0d2277d1de2f890067dcc0a,1371816739,38.906881,-78.246528,0
1296672,1296672,2020-06-21 12:12:32,3514865930894695,fraud_Stiedemann Ltd,food_dining,105.93,Christopher,Castaneda,M,1632 Cohen Drive Suite 639,High Rolls Mountain Park,NM,88325,32.9396,-105.8189,899,Naval architect,1967-08-30,483f52fe67fabef353d552c1e662974c,1371816752,33.619513,-105.130529,0
1296673,1296673,2020-06-21 12:13:36,2720012583106919,"fraud_Reinger, Weissnat and Strosin",food_dining,74.90,Joseph,Murray,M,42933 Ryan Underpass,Manderson,SD,57756,43.3526,-102.5411,1126,Volunteer coordinator,1980-08-18,d667cdcbadaaed3da3f4020e83591c83,1371816816,42.788940,-103.241160,0


In [28]:
american_df.columns

Index(['Unnamed: 0', 'trans_date_trans_time', 'cc_num', 'merchant', 'category',
       'amt', 'first', 'last', 'gender', 'street', 'city', 'state', 'zip',
       'lat', 'long', 'city_pop', 'job', 'dob', 'trans_num', 'unix_time',
       'merch_lat', 'merch_long', 'is_fraud'],
      dtype='object')

In [29]:
american_df["trans_date_trans_time"]=pd.to_datetime(american_df["trans_date_trans_time"])
american_df["merch_lat"]=american_df["merch_lat"].apply(float)
american_df["merch_long"]=american_df["merch_long"].apply(float)
american_df

,Unnamed: 0,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,city,state,zip,lat,long,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud
0,0,2019-01-01 00:00:18,2703186189652095,"fraud_Rippin, Kub and Mann",misc_net,4.97,Jennifer,Banks,F,561 Perry Cove,Moravian Falls,NC,28654,36.0788,-81.1781,3495,"Psychologist, counselling",1988-03-09,0b242abb623afc578575680df30655b9,1325376018,36.011293,-82.048315,0
1,1,2019-01-01 00:00:44,630423337322,"fraud_Heller, Gutmann and Zieme",grocery_pos,107.23,Stephanie,Gill,F,43039 Riley Greens Suite 393,Orient,WA,99160,48.8878,-118.2105,149,Special educational needs teacher,1978-06-21,1f76529f8574734946361c461b024d99,1325376044,49.159047,-118.186462,0
2,2,2019-01-01 00:00:51,38859492057661,fraud_Lind-Buckridge,entertainment,220.11,Edward,Sanchez,M,594 White Dale Suite 530,Malad City,ID,83252,42.1808,-112.2620,4154,Nature conservation officer,1962-01-19,a1a22d70485983eac12b5b88dad1cf95,1325376051,43.150704,-112.154481,0
3,3,2019-01-01 00:01:16,3534093764340240,"fraud_Kutch, Hermiston and Farrell",gas_transport,45.00,Jeremy,White,M,9443 Cynthia Court Apt. 038,Boulder,MT,59632,46.2306,-112.1138,1939,Patent attorney,1967-01-12,6b849c168bdad6f867558c3793159a81,1325376076,47.034331,-112.561071,0
4,4,2019-01-01 00:03:06,375534208663984,fraud_Keeling-Crist,misc_pos,41.96,Tyler,Garcia,M,408 Bradley Rest,Doe Hill,VA,24433,38.4207,-79.4629,99,Dance movement psychotherapist,1986-03-28,a41d7549acf90789359a9aa5346dcb46,1325376186,38.674999,-78.632459,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1296670,1296670,2020-06-21 12:12:08,30263540414123,fraud_Reichel Inc,entertainment,15.56,Erik,Patterson,M,162 Jessica Row Apt. 072,Hatch,UT,84735,37.7175,-112.4777,258,Geoscientist,1961-11-24,440b587732da4dc1a6395aba5fb41669,1371816728,36.841266,-111.690765,0
1296671,1296671,2020-06-21 12:12:19,6011149206456997,fraud_Abernathy and Sons,food_dining,51.70,Jeffrey,White,M,8617 Holmes Terrace Suite 651,Tuscarora,MD,21790,39.2667,-77.5101,100,"Production assistant, television",1979-12-11,278000d2e0d2277d1de2f890067dcc0a,1371816739,38.906881,-78.246528,0
1296672,1296672,2020-06-21 12:12:32,3514865930894695,fraud_Stiedemann Ltd,food_dining,105.93,Christopher,Castaneda,M,1632 Cohen Drive Suite 639,High Rolls Mountain Park,NM,88325,32.9396,-105.8189,899,Naval architect,1967-08-30,483f52fe67fabef353d552c1e662974c,1371816752,33.619513,-105.130529,0
1296673,1296673,2020-06-21 12:13:36,2720012583106919,"fraud_Reinger, Weissnat and Strosin",food_dining,74.90,Joseph,Murray,M,42933 Ryan Underpass,Manderson,SD,57756,43.3526,-102.5411,1126,Volunteer coordinator,1980-08-18,d667cdcbadaaed3da3f4020e83591c83,1371816816,42.788940,-103.241160,0


In [30]:
from preprocessing.preprocessing import ColumnNameTransformer,DateFeaturesTransformer
col_names={
    "amt":"de04_amount_transaction",
   "trans_date_trans_time": "requestdate",
    "merchant":"de42_aceptor_merchant_code",
    "cc_num":"de02_pan"
}
col_name_transformer=ColumnNameTransformer(col_names)
date_transformer=DateFeaturesTransformer()
american_df=col_name_transformer.transform(american_df)
american_df=date_transformer.transform(american_df)

In [31]:
from preprocessing.preprocessing import ColumnCalculator

metric_dict={'authorization_requests_count_and_value': {'windows': ['60min', '24h']}}
merchant_transformer=ColumnCalculator(metrics=metric_dict,entity="merchant")
american_df=merchant_transformer.transform(american_df)
american_df

,requestdate,Unnamed: 0,de02_pan,de42_aceptor_merchant_code,category,de04_amount_transaction,first,last,gender,street,city,state,zip,lat,long,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud,month,day_of_week,is_wknd,is_month_start,is_month_end,age,time_of_day,last_trans_date_trans_time,diff_time_last_trans_sec,last_merch_long,last_merch_lat,dist_with_last_merchant,merchant_authorization_tx_number_60min,merchant_authorization_tx_value_60min,merchant_authorization_tx_number_24h,merchant_authorization_tx_value_24h
0,2019-01-01 14:43:44,1251,4666314527820883145,fraud_Abbott-Rogahn,entertainment,74.63,Guy,Thompson,M,5812 Ramos Oval Suite 598,Ravenna,NE,68869,41.0233,-98.9041,2202,"Solicitor, Scotland",1974-06-21,dbe29230128b03874e77d7a046e93c28,1325429024,41.874726,-99.842215,0,1,2,0,1,0,44,day,2019-01-01 14:43:17,27,-94.593232,43.377055,583.987872,1.0,74.63,1.0,74.63
1,2019-01-01 18:09:49,1705,30044330818990,fraud_Abbott-Rogahn,entertainment,33.40,Allison,Ayala,F,87665 Karen Mill Apt. 586,Fort Myers,FL,33967,26.4722,-81.8122,224256,Paramedic,1985-08-29,525cb6be5e9fa5d6893da177096dfefd,1325441389,27.341741,-81.701833,0,1,2,0,1,0,33,day,2019-01-01 18:09:24,25,-84.806221,40.452772,383.171141,1.0,33.40,2.0,108.03
2,2019-01-01 22:52:02,2281,4623560839669,fraud_Abbott-Rogahn,entertainment,11.95,Vincent,Waller,M,9379 Vanessa Run Apt. 400,Lohrville,IA,51453,42.2619,-94.5566,695,Administrator,1954-07-15,7c84c6694da8d3fb17d876e8e53cc968,1325458322,41.658495,-93.860718,0,1,2,0,1,0,64,night,2019-01-01 22:51:49,13,-98.401488,30.209025,520.426951,1.0,11.95,3.0,119.98
3,2019-01-01 23:26:51,2354,560881134780,fraud_Abbott-Rogahn,entertainment,31.06,Audrey,Martinez,F,343 Joshua Views,Liberty Mills,IN,46946,41.0356,-85.7358,136,Soil scientist,1953-03-19,52a1fe875299ef10ecce0b4d7e3c9adf,1325460411,40.996279,-85.898931,0,1,2,0,1,0,65,night,2019-01-01 23:26:22,29,-71.045662,43.234019,1652.048362,2.0,43.01,4.0,151.04
4,2019-01-01 23:33:56,2366,4446368897795790,fraud_Abbott-Rogahn,entertainment,71.16,Ryan,Joseph,M,8850 Jessica Center,Atlantic,VA,23303,37.9073,-75.5089,741,"Geologist, engineering",1986-08-17,6e0514829a9849d7aa507492fa2f5175,1325460836,38.438757,-75.542933,0,1,2,0,1,0,32,night,2019-01-01 23:33:55,1,-73.635896,41.590781,231.530663,3.0,114.17,5.0,222.20
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1296670,2020-06-19 17:58:28,1292208,180048185037117,fraud_Zulauf LLC,personal_care,11.80,Mary,Wall,F,2481 Mills Lock,Plainfield,NJ,7060,40.6152,-74.4150,71485,Leisure centre manager,1974-07-19,832226cc2aaf34fee3272f2df03e6371,1371664708,41.396335,-74.471305,0,6,5,1,0,0,45,day,2020-06-19 17:58:18,10,-77.813544,38.115490,381.628143,3.0,16.47,7.0,291.29
1296671,2020-06-19 22:57:41,1292816,30074693890476,fraud_Zulauf LLC,personal_care,37.24,Kelsey,Richards,F,889 Sarah Station Suite 624,Holcomb,KS,67851,37.9931,-100.9893,2691,Arboriculturist,1993-08-16,d4896dc0b16bc1e1d26a481034652176,1371682661,38.077003,-100.564281,0,6,5,1,0,0,26,night,2020-06-19 22:56:44,57,-95.721145,41.761240,541.374919,1.0,37.24,7.0,256.26
1296672,2020-06-20 15:05:42,1294209,5388659976630693,fraud_Zulauf LLC,personal_care,18.42,Charles,Preston,M,5364 William Harbor,Seneca,WI,54654,43.2064,-90.9405,153,Quantity surveyor,1962-02-14,7d0b9009bcdd19fcea546cc4ddfb6e79,1371740742,43.613450,-90.262515,0,6,6,1,0,0,58,day,2020-06-20 15:05:09,33,-93.811977,37.928160,394.836327,1.0,18.42,6.0,96.75
1296673,2020-06-20 18:38:02,1294632,213141712584544,fraud_Zulauf LLC,personal_care,13.13,Margaret,Curtis,F,742 Oneill Shore,Florence,MS,39073,32.1530,-90.1217,19685,Fine artist,1984-12-24,851a99e5c0f299284130b8a437078357,1371753482,32.860663,-89.430287,0,6,6,1,0,0,35,day,2020-06-20 18:37:43,19,-121.149486,40.809718,3526.415097,1.0,13.13,3.0,68.79


In [32]:
card_transformer=ColumnCalculator(metrics=metric_dict,entity="card")
american_df=card_transformer.transform(american_df)
american_df

,requestdate,Unnamed: 0,de02_pan,de42_aceptor_merchant_code,category,de04_amount_transaction,first,last,gender,street,city,state,zip,lat,long,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud,month,day_of_week,is_wknd,is_month_start,is_month_end,age,time_of_day,last_trans_date_trans_time,diff_time_last_trans_sec,last_merch_long,last_merch_lat,dist_with_last_merchant,merchant_authorization_tx_number_60min,merchant_authorization_tx_value_60min,merchant_authorization_tx_number_24h,merchant_authorization_tx_value_24h,card_authorization_tx_number_60min,card_authorization_tx_value_60min,card_authorization_tx_number_24h,card_authorization_tx_value_24h
0,2019-01-01 12:47:15,1017,60416207185,"fraud_Jones, Sawayn and Romaguera",misc_net,7.27,Mary,Diaz,F,9886 Anita Drive,Fort Washakie,WY,82514,43.0048,-108.8964,1645,Information systems manager,1986-02-17,98e3dcf98101146a577f85a34e58feec,1325422035,43.974711,-109.741904,0,1,2,0,1,0,32,day,2019-01-01 12:46:45,30,-82.321074,38.628875,3046.340167,1.0,7.27,1.0,7.27,1.0,7.27,1.0,7.27
1,2019-01-02 08:44:57,2724,60416207185,fraud_Berge LLC,gas_transport,52.94,Mary,Diaz,F,9886 Anita Drive,Fort Washakie,WY,82514,43.0048,-108.8964,1645,Information systems manager,1986-02-17,498120fc45d277f7c88e3dba79c33865,1325493897,42.018766,-109.044172,0,1,3,0,0,0,32,day,2019-01-02 08:38:47,370,-96.773802,31.985329,1381.937000,1.0,52.94,4.0,270.47,1.0,52.94,2.0,60.21
2,2019-01-02 08:47:36,2726,60416207185,fraud_Luettgen PLC,gas_transport,82.08,Mary,Diaz,F,9886 Anita Drive,Fort Washakie,WY,82514,43.0048,-108.8964,1645,Information systems manager,1986-02-17,95f514bb993151347c7acdf8505c3d62,1325494056,42.961335,-109.157564,0,1,3,0,0,0,32,day,2019-01-02 08:45:14,142,-120.840904,44.255750,1300.478120,1.0,82.08,2.0,139.43,2.0,135.02,3.0,142.29
3,2019-01-02 12:38:14,2882,60416207185,fraud_Daugherty LLC,kids_pets,34.79,Mary,Diaz,F,9886 Anita Drive,Fort Washakie,WY,82514,43.0048,-108.8964,1645,Information systems manager,1986-02-17,4f0c1a14e0aa7eb56a490780ef9268c5,1325507894,42.228227,-108.747683,0,1,3,0,0,0,32,day,2019-01-02 12:36:59,75,-92.138649,30.624595,1852.296058,2.0,77.47,6.0,324.35,1.0,34.79,4.0,177.08
4,2019-01-02 13:10:46,2907,60416207185,fraud_Beier and Sons,home,27.18,Mary,Diaz,F,9886 Anita Drive,Fort Washakie,WY,82514,43.0048,-108.8964,1645,Information systems manager,1986-02-17,3b2ebd3af508afba959640893e1e82bc,1325509846,43.321745,-108.091143,0,1,3,0,0,0,32,day,2019-01-02 13:10:15,31,-81.453167,42.032224,2961.841700,1.0,27.18,7.0,423.27,2.0,61.97,4.0,196.99
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1296670,2020-06-20 21:04:59,1294934,4992346398065154184,"fraud_Berge, Kautzer and Harris",personal_care,60.47,Benjamin,Kim,M,920 Patrick Light,Mc Nabb,IL,61335,41.1730,-89.2187,532,Audiological scientist,1956-01-09,ad7dfdf0aaa36cd7985dd1f35ca0e2fc,1371762299,40.475395,-89.076105,0,6,6,1,0,0,64,night,2020-06-20 21:04:59,0,-92.271529,45.976671,354.978338,1.0,60.47,4.0,148.78,1.0,60.47,4.0,77.70
1296671,2020-06-21 00:41:01,1295369,4992346398065154184,fraud_Bernhard Inc,gas_transport,74.29,Benjamin,Kim,M,920 Patrick Light,Mc Nabb,IL,61335,41.1730,-89.2187,532,Audiological scientist,1956-01-09,6d427d735c9f9b2fd480f2c24b6525de,1371775261,40.743634,-89.553379,0,6,7,1,0,0,64,day,2020-06-21 00:40:49,12,-74.236227,43.153239,1703.234752,1.0,74.29,4.0,281.50,1.0,74.29,4.0,149.37
1296672,2020-06-21 02:47:59,1295587,4992346398065154184,"fraud_Reichert, Rowe and Mraz",shopping_net,246.56,Benjamin,Kim,M,920 Patrick Light,Mc Nabb,IL,61335,41.1730,-89.2187,532,Audiological scientist,1956-01-09,9814049bcc69fb31d81f4a907f2fe255,1371782879,40.215418,-88.682562,0,6,7,1,0,0,64,day,2020-06-21 02:47:48,11,-98.327036,26.654633,1068.880780,1.0,246.56,3.0,344.21,1.0,246.56,5.0,395.93
1296673,2020-06-21 08:04:28,1296206,4992346398065154184,fraud_Jewess LLC,shopping_pos,2.62,Benjamin,Kim,M,920